# RSD multipoles with SCOPE + Kaiser model validation

Uses the new `compute_xi_smu` function in SCOPE to measure ξ(s, μ) → ξ₀(s), ξ₂(s),
then overlays the linear-theory Kaiser prediction to validate the subvolume correction.

**Kaiser model** (Hamilton 1992, plane-parallel, distant-observer):
$$\xi_0(s) = \left(1 + \frac{2\beta}{3} + \frac{\beta^2}{5}\right) \xi(r)$$
$$\xi_2(s) = \left(\frac{4\beta}{3} + \frac{4\beta^2}{7}\right) \left[\xi(r) - \bar{\xi}(r)\right]$$
where $\beta = f/b$, $f = \Omega_m(z)^{0.55}$, and $\bar{\xi}(r) = \frac{3}{r^3}\int_0^r \xi(r')r'^2 dr'$.

In [1]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d
from scipy.optimize import minimize_scalar

SCOPE_PATH = "/cosma/apps/durham/dc-hick2/SCOPE/python"
PROJECT_ROOT = Path("../..").resolve()
if SCOPE_PATH not in sys.path:
    sys.path.insert(0, SCOPE_PATH)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import scope
from galform_analysis.config import Cosmology, get_snapshot_redshift
from galform_analysis.utils.read_galaxies import read_galaxy_arrays

## Parameters

In [2]:
IZ         = 271
SIM        = "L800"
MODEL      = "lc16"
BOXSIZE    = 542.16      # Mpc/h
K_TOTAL    = 1024        # total realisations
N_SUBVOLS  = 32          # realisations to load (increase for better stats)
MHALO_MIN  = 1e10        # M_sun/h
SEED       = 42

BASE_DIR = Path(f"/cosma5/data/durham/dc-hick2/Galform_Out/{SIM}/{MODEL}")

# Bin edges
R_BINS = np.logspace(np.log10(0.5), np.log10(100.0), 31)   # real space
S_BINS = np.logspace(np.log10(0.5), np.log10(80.0),  26)   # redshift space
N_MU_BINS = 100

# Kaiser fit scale range (Mpc/h)
FIT_SMIN, FIT_SMAX = 8.0, 60.0

print(f"iz={IZ}  n_subvols={N_SUBVOLS}/{K_TOTAL}")

iz=271  n_subvols=32/1024


## Load galaxies and apply RSD shift

In [ ]:
z_snap = get_snapshot_redshift(f"iz{IZ}")
h      = Cosmology.h
om     = Cosmology.OMEGA_M
ol     = Cosmology.OMEGA_L
ez     = np.sqrt(om * (1.0 + z_snap)**3 + ol)
hz     = 100.0 * h * ez   # H(z) in km/s/Mpc

print(f"z = {z_snap:.4f},  H(z) = {hz:.2f} km/s/Mpc,  f(z) = {(om*(1+z_snap)**3/ez**2)**0.55:.4f}")

# Select N_SUBVOLS random ivols
rng    = np.random.default_rng(SEED)
all_ivols = list(range(K_TOTAL))
ivols  = sorted(rng.choice(all_ivols, size=N_SUBVOLS, replace=False).tolist())

pos_real_chunks, pos_rsd_chunks, sv_chunks = [], [], []

iz_path = BASE_DIR / f"iz{IZ}"
for label, iv in enumerate(ivols):
    arr, meta = read_galaxy_arrays(
        iz_path,
        ivol=iv,
        mhalo_min=MHALO_MIN,
        fields=["vzgal"],
        include_positions=True,
    )
    x  = np.asarray(arr["x"],     dtype=np.float64)
    y  = np.asarray(arr["y"],     dtype=np.float64)
    z  = np.asarray(arr["z"],     dtype=np.float64)
    vz = np.asarray(arr["vzgal"], dtype=np.float64)

    # RSD displacement in Mpc/h
    ds   = (vz / hz) * h
    z_rsd = np.mod(z + ds, BOXSIZE)

    sv = np.full(len(x), label, dtype=np.int32)
    pos_real_chunks.append(np.column_stack([x, y, z]))
    pos_rsd_chunks.append(np.column_stack([x, y, z_rsd]))
    sv_chunks.append(sv)

coords_real = np.vstack(pos_real_chunks)
coords_rsd  = np.vstack(pos_rsd_chunks)
subvol_ids  = np.concatenate(sv_chunks)

print(f"Total galaxies loaded: {len(coords_real):,}")

z = 0.0000,  H(z) = 67.77 km/s/Mpc,  f(z) = 0.5223
Total galaxies loaded: 1,034,900


## Real-space ξ(r) with SCOPE

In [ ]:
res_xi = scope.compute_xi(
    coords_real, subvol_ids, R_BINS, BOXSIZE, K_TOTAL, N_SUBVOLS
)
r_mid = res_xi["r_mid"]
xi_r  = res_xi["xi"]
print("Real-space ξ(r) done")

## Redshift-space ξ(s, μ) with SCOPE

In [ ]:
res_smu = scope.compute_xi_smu(
    coords_rsd, subvol_ids, S_BINS, BOXSIZE, K_TOTAL, N_SUBVOLS,
    n_mu_bins=N_MU_BINS, mu_max=1.0,
)
s_mid  = res_smu["s_mid"]
mu_mid = res_smu["mu_mid"]
xi_smu = res_smu["xi_smu"]
xi0    = res_smu["xi0"]
xi2    = res_smu["xi2"]
print("RSD ξ(s, μ) done")

## Kaiser model prediction

In [ ]:
# Growth rate
omega_mz = om * (1.0 + z_snap)**3 / ez**2
f_z = omega_mz**0.55

# Interpolate ξ(r) onto fine grid for ξ̄(r) integration
# Extend to r=0 with ξ[0] (constant extrapolation inward)
r_fine  = np.linspace(0.0, r_mid[-1], 2000)
xi_interp = interp1d(r_mid, xi_r, kind="linear",
                     bounds_error=False, fill_value=(xi_r[0], 0.0))
xi_fine = xi_interp(r_fine)

# ξ̄(r) = 3/r³ ∫₀ʳ ξ(r') r'² dr'
integrand = xi_fine * r_fine**2
cum_int   = cumulative_trapezoid(integrand, r_fine, initial=0.0)
with np.errstate(divide="ignore", invalid="ignore"):
    xi_bar_fine = np.where(r_fine > 0, 3.0 / r_fine**3 * cum_int, xi_fine)

# Interpolate ξ̄ back onto s_mid
xi_bar_interp = interp1d(r_fine, xi_bar_fine, kind="linear",
                          bounds_error=False, fill_value=0.0)
xi_r_at_s    = xi_interp(s_mid)
xi_bar_at_s  = xi_bar_interp(s_mid)


def kaiser_multipoles(beta, xi_r_s, xi_bar_s):
    xi0_k = (1.0 + 2.0*beta/3.0 + beta**2/5.0) * xi_r_s
    xi2_k = (4.0*beta/3.0 + 4.0*beta**2/7.0) * (xi_r_s - xi_bar_s)
    return xi0_k, xi2_k


# Fit β on the specified scale range using both ξ₀ and ξ₂
mask = (s_mid >= FIT_SMIN) & (s_mid <= FIT_SMAX)

def chi2(beta):
    xi0_k, xi2_k = kaiser_multipoles(beta, xi_r_at_s[mask], xi_bar_at_s[mask])
    r0 = (xi0[mask] - xi0_k) / (np.abs(xi0[mask]) + 0.01)
    r2 = (xi2[mask] - xi2_k) / (np.abs(xi2[mask]) + 0.01)
    return np.sum(r0**2 + r2**2)

result  = minimize_scalar(chi2, bounds=(0.05, 2.0), method="bounded")
beta_fit = result.x
b_fit    = f_z / beta_fit

print(f"f(z)    = {f_z:.4f}")
print(f"β fit   = {beta_fit:.4f}")
print(f"b fit   = {b_fit:.4f}")

xi0_kaiser, xi2_kaiser = kaiser_multipoles(beta_fit, xi_r_at_s, xi_bar_at_s)

## Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# ── Panel 1: ξ(s, μ) wedge map ──────────────────────────────────────────────
ax = axes[0]
vlim = 1.0
im = ax.pcolormesh(
    s_mid, mu_mid, xi_smu.T,
    vmin=-vlim, vmax=vlim, cmap="RdBu_r",
)
fig.colorbar(im, ax=ax, label=r"$\xi(s,\mu)$")
ax.set_xlabel(r"$s$ [Mpc/h]")
ax.set_ylabel(r"$\mu = |\Delta z|/s$")
ax.set_xscale("log")
ax.set_title(r"$\xi(s,\mu)$")

# ── Panel 2: monopole ξ₀(s) ──────────────────────────────────────────────────
ax = axes[1]
ax.plot(s_mid, s_mid**2 * xi0,        color="C0", label=r"SCOPE $\xi_0(s)$")
ax.plot(s_mid, s_mid**2 * xi0_kaiser, color="C0", ls="--", label=r"Kaiser ($\beta={:.3f}$)".format(beta_fit))
ax.plot(r_mid, r_mid**2 * xi_r,       color="k",  ls=":",  label=r"real-space $\xi(r)$", alpha=0.6)
ax.axvline(FIT_SMIN, color="gray", lw=0.8, ls=":")
ax.axvline(FIT_SMAX, color="gray", lw=0.8, ls=":")
ax.set_xscale("log")
ax.set_xlabel(r"$s$ [Mpc/h]")
ax.set_ylabel(r"$s^2\,\xi_0(s)$  [$(\mathrm{Mpc}/h)^2$]")
ax.set_title(r"Monopole $\xi_0(s)$")
ax.legend(fontsize=8)

# ── Panel 3: quadrupole ξ₂(s) ────────────────────────────────────────────────
ax = axes[2]
ax.plot(s_mid, s_mid**2 * xi2,        color="C1", label=r"SCOPE $\xi_2(s)$")
ax.plot(s_mid, s_mid**2 * xi2_kaiser, color="C1", ls="--", label=r"Kaiser ($\beta={:.3f}$)".format(beta_fit))
ax.axhline(0, color="k", lw=0.5)
ax.axvline(FIT_SMIN, color="gray", lw=0.8, ls=":")
ax.axvline(FIT_SMAX, color="gray", lw=0.8, ls=":")
ax.set_xscale("log")
ax.set_xlabel(r"$s$ [Mpc/h]")
ax.set_ylabel(r"$s^2\,\xi_2(s)$  [$(\mathrm{Mpc}/h)^2$]")
ax.set_title(r"Quadrupole $\xi_2(s)$")
ax.legend(fontsize=8)

fig.suptitle(
    f"L800 lc16 iz{IZ} (z={z_snap:.2f}), "
    f"n={N_SUBVOLS}/{K_TOTAL} sub-vols, "
    r"$\beta_\mathrm{fit}=$" + f"{beta_fit:.3f}, "
    r"$b_\mathrm{fit}=$" + f"{b_fit:.3f}",
    fontsize=10,
)
fig.tight_layout()

out_dir = Path("_plots/scope_rsd_kaiser")
out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / f"scope_rsd_kaiser_iz{IZ}_n{N_SUBVOLS}.png", dpi=150)
plt.show()
print("Saved to", out_dir)

## β sweep: Kaiser predictions for a range of β values

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
betas = [0.2, 0.3, 0.4, beta_fit, 0.6, 0.8]
cmap  = plt.cm.viridis
colors = cmap(np.linspace(0.1, 0.9, len(betas)))

for ax, (xi_meas, xi_k_func, label) in zip(
    axes,
    [
        (xi0, lambda b: kaiser_multipoles(b, xi_r_at_s, xi_bar_at_s)[0], r"$\xi_0(s)$"),
        (xi2, lambda b: kaiser_multipoles(b, xi_r_at_s, xi_bar_at_s)[1], r"$\xi_2(s)$"),
    ],
):
    ax.plot(s_mid, s_mid**2 * xi_meas, "k-", lw=2, label="SCOPE measured")
    for beta_i, col in zip(betas, colors):
        lw  = 2.5 if np.isclose(beta_i, beta_fit) else 1.0
        ls  = "-" if np.isclose(beta_i, beta_fit) else "--"
        lab = rf"$\beta={beta_i:.2f}$" + (" (fit)" if np.isclose(beta_i, beta_fit) else "")
        ax.plot(s_mid, s_mid**2 * xi_k_func(beta_i), color=col, lw=lw, ls=ls, label=lab)
    ax.axhline(0, color="k", lw=0.5)
    ax.set_xscale("log")
    ax.set_xlabel(r"$s$ [Mpc/h]")
    ax.set_ylabel(rf"$s^2\,{label.strip('$')}$  [$(\mathrm{{Mpc}}/h)^2$]")
    ax.set_title(label)
    ax.legend(fontsize=7, ncol=2)

fig.suptitle(
    f"Kaiser β sweep — L800 lc16 iz{IZ} (z={z_snap:.2f}), "
    f"n={N_SUBVOLS}/{K_TOTAL} sub-vols",
    fontsize=10,
)
fig.tight_layout()
fig.savefig(out_dir / f"scope_rsd_kaiser_bsweep_iz{IZ}_n{N_SUBVOLS}.png", dpi=150)
plt.show()

## Summary

In [ ]:
print("="*55)
print(f"  Snapshot    : iz{IZ}  (z = {z_snap:.4f})")
print(f"  Sub-volumes : {N_SUBVOLS} / {K_TOTAL}")
print(f"  Galaxies    : {len(coords_real):,}")
print(f"  Mhalo min   : {MHALO_MIN:.0e} M_sun/h")
print()
print(f"  f(z)        : {f_z:.4f}")
print(f"  β fit       : {beta_fit:.4f}")
print(f"  b fit       : {b_fit:.4f}")
print(f"  f/b = β     : {f_z/b_fit:.4f} (consistency check)")
print("="*55)